# TCGA Data Access

In this session, we will learn how to use the R programming language to access data from TCGA (The Cancer Genome Atlas).

## Make Your Own Copy

In Colab, sign in to your Google account and choose **File → Save a copy in Drive**. Open the saved copy from Google Drive. You can edit cells, try your own code, and save your work there.

## Installation of Packages

For a CRAN package, `install.packages("package_name")` is usually all you need. This notebook also uses Bioconductor packages, which it installs with `BiocManager::install()`.

Colab runs R on Ubuntu Linux. The setup cell detects the Ubuntu release and uses Posit Package Manager to request matching precompiled packages when available. This speeds up setup by avoiding compilation for those packages. On other systems, the cell uses the standard CRAN and Bioconductor repositories.

Run the setup cell at the start of each new Colab runtime. It checks which packages are already available and installs only the missing ones. The next cell loads the packages for use in the notebook.



In [ ]:
# Request Posit's precompiled packages on Ubuntu (including Colab).
if (file.exists("/etc/os-release") && any(grepl('^ID="?ubuntu"?$', readLines("/etc/os-release")))) {
  ubuntu_codename <- system("lsb_release -cs", intern = TRUE)
  host <- paste0("https://", "packagemanager", ".", "posit", ".", "co")
  p3m_cran <- paste(host, "cran", "__linux__", ubuntu_codename, "latest", sep = "/")
  p3m_bioc <- paste(host, "bioconductor", "__linux__", ubuntu_codename, "latest", sep = "/")
  options(
    HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
      paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
    repos = c(CRAN = p3m_cran),
    BioC_mirror = p3m_bioc
  )
} else {
  options(repos = c(CRAN = "https://cloud.r-project.org"))
}

cran_packages <- c("BiocManager", "tidyverse", "knitr")

missing_cran <- cran_packages[
  !vapply(cran_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_cran) > 0) {
  install.packages(missing_cran)
}

bioc_packages <- c("TCGAbiolinks", "SummarizedExperiment")

missing_bioc <- bioc_packages[
  !vapply(bioc_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_bioc) > 0) {
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE)
}



You can load the installed packages:



In [ ]:
library(knitr)
library(SummarizedExperiment)
library(TCGAbiolinks)
library(tidyverse)


The helper below renders data frames as HTML tables in Colab and adds horizontal scrolling for wide tables. It also works when this Quarto document is rendered locally.



In [ ]:
#@title Table display setup
display_table <- function(data, caption = NULL) {
  table_html <- knitr::kable(
    as.data.frame(data),
    format = "html",
    align = "l",
    caption = caption,
    escape = TRUE,
    table.attr = 'class="colab-data-table"'
  )

  scrollable_table <- paste0(
    "<style>",
    ".colab-table-wrapper {max-width:100%; overflow-x:auto; margin:1em 0;}",
    ".colab-data-table {border-collapse:collapse; width:max-content; ",
    "font-size:0.9em; white-space:nowrap;}",
    ".colab-data-table caption {text-align:left; font-weight:600; ",
    "padding:0 0 0.6em 0;}",
    ".colab-data-table th, .colab-data-table td {border:1px solid #777; ",
    "padding:6px 10px; text-align:left; vertical-align:top;}",
    ".colab-data-table th {background:rgba(127,127,127,0.22); ",
    "font-weight:600;}",
    ".colab-data-table tbody tr:nth-child(even) {",
    "background:rgba(127,127,127,0.08);}",
    "</style>",
    '<div class="colab-table-wrapper">',
    table_html,
    "</div>"
  )

  if (isTRUE(getOption("knitr.in.progress"))) {
    return(knitr::asis_output(scrollable_table))
  }

  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_html(scrollable_table)
    return(invisible(data))
  }

  # Fallback for other R notebook environments.
  structure(scrollable_table, class = c("html", "character"))
}


## Setting up working directory

A working directory is the folder where R reads and saves files. When this notebook runs in Colab, opening it from GitHub downloads only the notebook itself. The following cell clones the complete repository, including the prepared `.rds` datasets, and selects it as the working directory. When working locally, your existing working directory is left unchanged.



In [ ]:
repo_dir <- "/content/MedInfNetworks2026"

if (dir.exists("/content") && !dir.exists(repo_dir)) {
  system2(
    "git",
    c(
      "clone",
      "https://github.com/tanevskilab/MedInfNetworks2026.git",
      repo_dir
    )
  )
}

if (dir.exists(repo_dir)) {
  setwd(repo_dir)
}

getwd()



## Accessing TCGA Data

TCGA is a massive public resource containing genomic and clinical data for dozens of cancer types. We interact with the TCGA data through the GDC (Genomic Data Commons) knowledge base by using the `TCGAbiolinks` package or by accessing the data portal in a browser.

### Access via R and `TCGAbiolinks`

With the `TCGAbiolinks` package loaded, we retrieve a list of all available projects from the GDC database. Since TCGA projects start with the prefix "TCGA-", we can filter the full list to focus only on cancer-specific projects. Displaying the project names helps identifying for which cancers data is available.



In [ ]:
all_projects <- getGDCprojects()

# Filter for TCGA projects
tcga_projects <- all_projects[grepl("^TCGA-", all_projects$project_id), ]

# Show available cancer types
display_table(
  head(tcga_projects[, c("project_id", "name")]),
  caption = "Available TCGA Cancer Types"
)



In this tutorial, we focus on **Breast Cancer** (TCGA-BRCA) in **females** as an example. Let’s look at the available data categories for BRCA:



In [ ]:
project_summary <- getProjectSummary(project = "TCGA-BRCA")

display_table(
  project_summary$data_categories,
  caption = "Available Data Categories for BRCA"
)



#### Building a GDC Query

We want to take a look at RNA-seq data, and to do this we must search the GDC database for the specific files we want. In TCGAbiolinks, searching the GDC is done by building a query. A query tells the GDC exactly which files you are interested in.

A query for BRCA RNA-seq data must specify the following:

-   which project? (TCGA-BRCA)

Optional specifications are:

-   which data category? (Transcriptome Profiling)

-   which data type? (Gene Expression Quantification)

-   which workflow? (STAR - Counts)

If you are unsure which values to use, the official TCGAbiolinks [vignette](https://bioconductor.org/packages/release/bioc/vignettes/TCGAbiolinks/inst/doc/query.html) provides a helpful description of supported data.type and workflow.type options.



In [ ]:
# Database search
brca_samples <- GDCquery(
  project = "TCGA-BRCA",
  data.category = "Transcriptome Profiling",
  data.type = "Gene Expression Quantification",
  workflow.type = "STAR - Counts"
)

# Extract sample metadata
sample_info <- getResults(brca_samples)
display_table(head(sample_info, 10), caption = "Sample Metadata (first 10 rows)")



The resulting metadata includes useful information, such as:

-   file, sample, and patient IDs

-   sample type (e.g., tumor tissue vs. normal tissue)

-   technical details and sequencing information

This metadata tells us what files exist, but it does not include some important patient-level information, such as reported sex, age, or diagnosis. These details are stored separately in the clinical dataset.

#### Accessing Clinical Data



In [ ]:
brca_clin <- GDCquery_clinic(project = "TCGA-BRCA", type = "clinical")
brca_clin$reported_sex <- tolower(trimws(brca_clin$sex_at_birth))
brca_clin$patient_id <- brca_clin$submitter_id

display_table(head(brca_clin, 10), caption = "Clinical Data (first 10 rows)")



The clinical table typically includes:

-   reported sex

-   age at diagnosis

-   vital status (alive/deceased)

-   tumor stage

-   etc.

First, we verify whether the clinical data contains any patients without a reported sex entry:



In [ ]:
sum(is.na(brca_clin$reported_sex))



If one or more entries have missing values, we remove those rows to avoid problems later:



In [ ]:
brca_clin <- brca_clin %>%
  filter(!is.na(reported_sex)) 



Next, we inspect how many males and females are present in the cleaned clinical dataset:



In [ ]:
display_table(count(brca_clin, reported_sex, name = "Patients"), caption = "Patients by Reported Sex")



This gives us the number of unique male and female patients in TCGA-BRCA.

#### Merging Reported Sex Information Into Sample Metadata

To filter samples by reported sex later, we need reported sex information added directly to the `sample_info` table, which currently only contains sample-level metadata.

We merge the two tables using:

-   "cases.submitter_id" (in `sample_info`)

-   "patient_id" (in `brca_clin`)

These are equivalent patient identifiers.



In [ ]:
sample_info <- sample_info %>%
  inner_join(
    brca_clin %>% select(patient_id, reported_sex),
    by = c("cases.submitter_id" = "patient_id")
  )



The inner join keeps samples whose patient ID matches a clinical record with a reported sex.

After merging, we check how many samples belong to each reported sex:



In [ ]:
display_table(count(sample_info, reported_sex, name = "Samples"), caption = "Samples by Reported Sex")


> **Exercise**
>
> Why are the numbers now different? Before you continue, try to think about why this might be.


###### Show answer


This does not mean TCGA suddenly contains more patients, instead it means some patients contributed multiple samples, each counted separately. To verify this, we identify patients that appear more than once:


In [ ]:
sample_info %>%
  add_count(cases.submitter_id) %>%  # adds column 'n' (number of appearances)
  filter(n > 1) %>%                  # keep only duplicated patient IDs
  arrange(cases.submitter_id) %>%
  select(cases.submitter_id, sample.submitter_id, sample_type, reported_sex, n) %>%
  head() %>%
  display_table(caption = "Patients With Multiple Samples")


This reveals that many patients have two samples:

-   Primary Tumor

-   Solid Tissue Normal

A single patient may contribute multiple samples. Therefore, the reported sex counts increase because we now count samples, not patients.


#### Filtering to Keep Only Female Samples

Once we confirm the reported sex assignment, we can keep only female samples:


In [ ]:
sample_info <- sample_info %>% filter(reported_sex == "female")



From this point forward, all analyses will be limited to female BRCA samples, which is common since male breast cancer biology differs significantly and is extremely rare.

#### Selecting a Subset of Samples to Download

Now we are finally ready to download real RNA-seq data from TCGA. But downloading the full BRCA dataset would take a very long time. Therefore, for teaching purposes, we only download a small subset.

We choose:

-   10 Primary Tumor samples

-   10 Solid Tissue Normal samples

from 10 patients total, meaning we have one tumor and control sample per patient.

We extract the sample barcodes from `sample_info`, which already contains the reported sex and clinical metadata we merged earlier.



In [ ]:
# Identify patients with exactly 2 samples
paired_patients_info <- sample_info %>%
  add_count(cases.submitter_id) %>%
  filter(n == 2) %>%
  distinct(cases.submitter_id, .keep_all = TRUE)

# Subset sample_info to include only these paired patients
paired_samples <- sample_info %>%
  filter(cases.submitter_id %in% paired_patients_info$cases.submitter_id)

# Sanity check – ensure each patient has exactly 1 tumor and 1 normal
valid_paired_patients <- paired_samples %>%
  group_by(cases.submitter_id) %>%
  summarize(
    n_samples = n(),
    n_tumor = sum(sample_type == "Primary Tumor"),
    n_normal = sum(sample_type == "Solid Tissue Normal")
  ) %>%
  filter(n_tumor == 1 & n_normal == 1) %>%
  pull(cases.submitter_id)  # extract the patient IDs as a vector

# Keep only the first 10 patients for quick download
paired_samples <- paired_samples %>%
  filter(cases.submitter_id %in% valid_paired_patients[1:10])


After selecting the 20 barcodes, we build a new query that fetches only these specific files.

> **Exercise**
>
> Try it by yourself. Tip: To specify the samples, you need to add the parameter `barcode` in `GDCquery`.


###### Show answer


In [ ]:
#| code-fold: true
# Query only these samples
brca_query_small <- GDCquery(
  project = "TCGA-BRCA",
  data.category = "Transcriptome Profiling",
  data.type = "Gene Expression Quantification",
  workflow.type = "STAR - Counts",
  barcode = paired_samples$cases
)



#### Downloading RNA-seq Gene Count Data

Once the query is built, we run the GDC download function, which retrieves 20 RNA-seq count files associated with our selected barcodes.



In [ ]:
#| eval: false

# Download the files 
GDCdownload(brca_query_small)



#### Preparing the Data with GDCprepare

After downloading, GDCprepare() automatically:

-   loads the downloaded files

-   parses metadata and merges it with sample information

-   constructs a SummarizedExperiment object

-   organizes:

    -   the gene expression count matrix

    -   sample metadata (columns)

    -   gene annotations / features (rows)



In [ ]:
brca_data_small <- GDCprepare(brca_query_small)


In [ ]:
brca_data_small



Printing the object shows:

-   60660 genes (rows)

-   20 samples (columns)

The SummarizedExperiment object has three interconnected components:

1.  Expression matrix (assay)
    -   A numeric matrix of genes × samples

    -   Each entry is typically an integer count of reads mapping to a gene
2.  Sample metadata (colData)
    -   A table describing each sample (columns in the matrix)
3.  Feature (gene) metadata (rowData)
    -   A table describing each gene (rows in the matrix)



In [ ]:
expr_matrix <- assay(brca_data_small)        
sample_metadata <- colData(brca_data_small)  
feature_metadata <- rowData(brca_data_small) 



#### Exploring Gene Metadata and Filtering for Protein-Coding Genes

The feature_metadata (rowData of the SummarizedExperiment) contains information about each gene. It’s good practice to explore the metadata before analysis.



In [ ]:
display_table(head(feature_metadata), caption = "Gene Metadata")



Here you can see columns like:

-   gene_id: Ensembl ID

-   gene_name: gene symbol

-   gene_type: type of gene (protein_coding, lncRNA, pseudogene, etc.)

Filter to keep only protein-coding genes:



In [ ]:
# Keep only protein-coding genes
brca_data_small_pc <- brca_data_small[rowData(brca_data_small)$gene_type == "protein_coding", ]

# Check how many genes remain
brca_data_small_pc


After this step, the SummarizedExperiment contains only protein-coding genes.

#### Splitting Into Tumor and Normal SummarizedExperiments

> **Exercise**
>
> Now try to divide the dataset into two smaller objects yourself:
>
> -   one containing only the 10 tumor samples
>
> -   one containing only the 10 normal samples


###### Show answer


In [ ]:
#| code-fold: true
tumor_SE <- brca_data_small_pc[, colData(brca_data_small_pc)$sample_type == "Primary Tumor"]
normal_SE <- brca_data_small_pc[, colData(brca_data_small_pc)$sample_type == "Solid Tissue Normal"]



#### Saving the Dataset



In [ ]:
#| eval: false

saveRDS(brca_data_small_pc, file = "brca_data_small.rds")



### Access via the Webportal

There is already a step-by-step guide on how to use the website [here](https://bioinformatics.ccr.cancer.gov/docs/btep-coding-club/CC2024/TCGA/TCGA_download/#steps-to-download-data-from-the-gdc). You can try to follow along the guide and select the filters we used in this notebook to download the data by choosing download cart in the last step.
